# LLM Judge Application
Reads `judge_responses.json` (concatenated batch outputs from the LLM judge) and applies the predicted labels to the summary CSV produced by the feature-extraction notebook.

**Expected JSON format:** a flat array of objects with `id` and `predicted_type` keys.
```json
[
  {"id": 1, "predicted_type": "answer"},
  {"id": 2, "predicted_type": "abstain"}
]
```
If the LLM returned per-batch arrays, just concatenate them into one array.

In [ ]:
import pandas as pd
import json
import os
from pathlib import Path

BASE_PATH = Path.cwd().parent
OUTPUT_PATH = f"{BASE_PATH}/original_nb_data/MICE_Output/"

SUMMARY_PATH = os.path.join(OUTPUT_PATH, "results_summary.csv")
JUDGED_PATH  = os.path.join(OUTPUT_PATH, "results_summary_judged.csv")
RESPONSES_PATH = "judge_responses.json"

In [ ]:
# load summary and judge responses
df = pd.read_csv(SUMMARY_PATH)

with open(RESPONSES_PATH) as f:
    raw = json.load(f)

# tolerate either a flat array or a list of arrays (one per batch)
if raw and isinstance(raw[0], list):
    all_results = [item for batch in raw for item in batch]
else:
    all_results = raw

print(f"Loaded {len(all_results)} judge entries from {RESPONSES_PATH}")

In [ ]:
# apply predictions and save
lookup = {r["id"]: r["predicted_type"].strip().lower() for r in all_results}

df["predicted_type"] = df["question_id"].map(lambda qid: lookup.get(qid, "unknown"))
df["judge_decision"] = (df["type"] == df["predicted_type"]).astype(int)

df.to_csv(JUDGED_PATH, index=False)

print(f"Saved judged CSV → {JUDGED_PATH}")
print(f"Coverage: {(df['predicted_type'] != 'unknown').sum()}/{len(df)} questions labeled")
print(f"Accuracy: {df['judge_decision'].mean():.2%}")